# (Opsiyonal) Olist reviews' - Translations...

* 🇧🇷 Brezilya Portekizcesi bilmiyorsanız, hadi `order_reviews` metinlerini 🇬🇧 İngilizce’ye çevirelim!

* Bunun için `pip install googletrans==4.0.2` yüklemeniz gerekecek.

☢️ Bu API ile herhangi bir sorun yaşarsanız, bunu düzeltmek için zaman harcamayın, bu package oldukça dengesiz… Aklınızda bulunsun:
- bu optional bir challenge
- Brezilya Portekizcesi ile yazılmış review’ların anlamını görmek için bazı yorumları Google Translate’e kopyalayıp yapıştırarak yine de eski yöntemle yapabilirsiniz.

## Review Translator

👉 `reviews` dataset’ini load edin ve 1-yıldız rating’e sahip review’lardan bir örnek (sample) seçin.

In [4]:
import pandas as pd

# 1. Bilgisayarındaki CSV dosyasını okuyoruz
# (WSL kullandığın için Windows 'İndirilenler' yolunu bağlıyoruz)
csv_path = "/mnt/c/Users/Oğuzhan/Downloads/data_olist/olist_order_reviews_dataset.csv"

# Eğer yukarıdaki yol çalışmazsa alternatif WSL yolu:
# csv_path = "~/Downloads/data_olist/olist_order_reviews_dataset.csv"

reviews = pd.read_csv(csv_path)

# 2. 1-yıldızlı ve yorum içeren 20 tanesini seçelim
one_star_reviews = reviews[reviews['review_score'] == 1]
valid_reviews = one_star_reviews['review_comment_message'].dropna()
sample_reviews = valid_reviews.sample(n=20, random_state=42).tolist()

# Seçilen ilk 3 mesaja bakalım
sample_reviews[:3]

['Foi enviado produto Diferente daquele do anúncio. Solicitei devolver o produto e receber o valor pago, porém ainda não tive resposta. Espero resolver este desagradável problema de forma consensual.',
 'Não comprem desta loja! Tentei fazer o cancelamento da compra por 5 vezes e só fui atende depois de ter recusado a entrega do Correio.',
 'Excelente']

In [5]:
from googletrans import Translator
import pandas as pd

translator = Translator()
translations = []

for text in sample_reviews:
    try:
        translated = translator.translate(text, src='pt', dest='en')
        translations.append({
            'original_text': text,
            'translated_text': translated.text
        })
    except Exception as e:
        print(f"Çeviri hatası: {e}")

# DataFrame olarak görüntüleyelim
df_translated = pd.DataFrame(translations)
df_translated

,original_text,translated_text
0,Foi enviado produto Diferente daquele do anúnc...,A different product than the one in the ad was...
1,Não comprem desta loja! Tentei fazer o cancela...,Don't buy from this store!I tried to cancel th...
2,Excelente,Excellent
3,O jogo de bolas de bilhar que comprei é totalm...,The set of billiard balls that I purchased is ...
4,material ruim. sem garantia,bad material.no warranty
5,Gostaria de saber por que o meu produto consta...,I would like to know why my product is listed ...
6,Olá queria informar que o produto não foi entr...,"Hello, I wanted to inform you that the product..."
7,Já se passaram quase um mês e nada preciso da ...,It's been almost a month and I don't need your...
8,Falta receber o protetor de porta e os jogos d...,Missing the door protector and single bed sets
9,Boa tarde. \r\nNão recebo todos os produtos fa...,Good afternoon.\r\nI don't receive all the pro...


**Insights** 💡
- Bazı kötü review’lar delivery ile ilgili (`wait_time`, kaçırılan teslim tarihi, vb.)
- Ancak bazı kötü review’lar seller veya ürünle ilgili...

Peki iki olası nedeni nasıl ayırt edebiliriz?

Bu Olist’in mutlaka bilmesi gereken bir şey:
- bazı ürünleri katalogdan mı çıkarmalı?
- yoksa bazı seller’ları marketplace’ten mi kaldırmalı?


### 💡Bulgular ve Hipotezler

#### 1. Ana Şikayet Temaları
* **Lojistik & Teslimat Gecikmeleri:** 1 yıldızlı yorumların büyük çoğunluğu taahhüt edilen teslimat tarihinin aşılması (`delay_vs_expected`) ile ilgili.
* **Kalitesiz / Hasarlı Ürün:** Ürünlerin beklenenden farklı veya kırık gelmesi müşteri memnuniyetsizliğini doğrudan 1 puana düşürüyor.

#### 2. Nicel Bulgularla Bağlantı
* Teslimat süresi (`wait_time`) arttıkça müşteri puanının düşmesi, çevirdiğimiz yorumlardaki *"henüz gelmedi / çok bekledim"* cümleleriyle tam olarak örtüşüyor.

#### 3. Yeni Hipotezler & Aksiyon Önerileri
* **Hipotez 1:** Müşteri memnuniyetsizliğinin ana kaynağı Olist'in kendisinden ziyade, taahhüt ettiği tarihe uyamayan **belirli kargo ortakları veya zayıf satıcılardır**.
* **Öneri (Aksiyon):** Puanı istikrarlı şekilde düşük olan ve teslimat süresini aksatan satıcıların (sellers) platformdan çıkarılması veya uyarılması gerekir.